# Product Recommendation Engine Workbook

This notebook is designed for beginners. It builds a product recommendation
engine for a generic online shop using customer-product interaction data.

The workflow follows the recommendation-engine ideas from the product
recommendation page: browsing, clicks, purchases, ratings/reviews, product
information.

In this workbook you will:

1. Load sample CSV files.
2. Convert customer events into an implicit-feedback user-item matrix.
3. Apply BM25 re-weighting.
4. Train an Alternating Least Squares matrix-factorization model.
5. Use Optuna to search hyperparameters.
6. Save recommendation and evaluation outputs.

## Before You Start

If packages are missing, open a terminal in this folder and run:

```bash
pip install -r requirements.txt
```

Then restart the notebook kernel and run the cells again.

## Cell 1: Import Packages

Run this cell first. It imports standard analysis packages and the local
helper code in the `code/` folder.

The notebook can run a small educational ALS fallback if `implicit` is not
installed, but install the requirements for the intended BM25, ALS, and
Optuna workflow.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd()
CODE_DIR = BASE_DIR / "code"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MPL_CACHE_DIR = OUTPUT_DIR / ".matplotlib"
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from product_recommender import (
    apply_bm25_weighting,
    build_user_item_matrix,
    evaluate_recommendations,
    fit_als_model,
    get_recommendations_for_user,
    get_similar_items,
    load_sample_data,
    prepare_interaction_strengths,
    run_optuna_search,
    train_test_split_by_user,
)

missing_packages = [
    package_name
    for package_name in ["implicit", "optuna"]
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print("Missing optional/required modelling packages:", missing_packages)
    print("Run this in a terminal opened in this folder: pip install -r requirements.txt")
else:
    print("All modelling packages are available.")

## Cell 2: File Locations and User Settings

Basic users usually only need to edit the settings below.

`N_OPTUNA_TRIALS` controls how many hyperparameter combinations Optuna
tests. Increase it for a deeper search, or reduce it if the notebook is
taking too long.

In [ ]:
INTERACTIONS_PATH = BASE_DIR / "customer_product_interactions.csv"
CATALOG_PATH = BASE_DIR / "product_catalog.csv"
CUSTOMER_SEGMENTS_PATH = BASE_DIR / "customer_segments.csv"

TOP_N_RECOMMENDATIONS = 10
N_OPTUNA_TRIALS = 20
RANDOM_STATE = 42

DEFAULT_PARAMS = {
    "factors": 32,
    "regularization": 0.05,
    "alpha": 40.0,
    "iterations": 12,
    "bm25_k1": 100.0,
    "bm25_b": 0.8,
}

print("Base folder:", BASE_DIR)
print("Outputs will be saved to:", OUTPUT_DIR)

## Cell 3: Load the Sample CSV Files

The main modelling file is `customer_product_interactions.csv`.

Each row is one customer-product event, such as a view, click, basket add,
purchase, or review. Matrix factorization uses the interaction file. The
product catalog is used to make recommendation outputs readable.

In [ ]:
interactions, catalog = load_sample_data(BASE_DIR)
customer_segments = pd.read_csv(CUSTOMER_SEGMENTS_PATH)

print("Interaction rows:", len(interactions))
print("Products:", catalog["item_id"].nunique())
print("Customers:", interactions["user_id"].nunique())

display(interactions.head())
display(catalog.head())
display(customer_segments.head())

## Cell 4: Build the User-Item Matrix

Recommendation models need a matrix where:

- rows are customers
- columns are products
- values represent confidence that a customer is interested in a product

The notebook first converts event types into strengths. For example, a
purchase is stronger evidence than a product view. Then it aggregates all
events for each customer-product pair.

In [ ]:
prepared_interactions = prepare_interaction_strengths(interactions)
user_item_matrix, aggregated_interactions, mappings = build_user_item_matrix(
    prepared_interactions
)

aggregated_interactions.to_csv(
    OUTPUT_DIR / "aggregated_user_item_interactions.csv", index=False
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Non-zero user-item pairs:", user_item_matrix.nnz)
print("Matrix sparsity:", round(1 - user_item_matrix.nnz / np.prod(user_item_matrix.shape), 4))

display(aggregated_interactions.head())

## Cell 5: Train/Test Split

To evaluate the recommender, we hide a few known customer-product
interactions from the model. After training, we check whether the model
recommends those hidden products.

In [ ]:
train_matrix, test_matrix = train_test_split_by_user(
    user_item_matrix,
    test_fraction=0.2,
    min_items_for_test=4,
    random_state=RANDOM_STATE,
)

print("Training interactions:", train_matrix.nnz)
print("Held-out test interactions:", test_matrix.nnz)

## Cell 6: Baseline BM25 plus ALS Model

BM25 re-weighting reduces the impact of products that almost everyone
interacts with and customers who interact with many products.

ALS then decomposes the sparse customer-product matrix into two lower
dimensional matrices: one for customers and one for products. The algorithm
alternates between fixing one matrix and optimizing the other, which makes
it practical for sparse recommendation data.

In [ ]:
weighted_train_default = apply_bm25_weighting(
    train_matrix,
    k1=DEFAULT_PARAMS["bm25_k1"],
    b=DEFAULT_PARAMS["bm25_b"],
)

baseline_model = fit_als_model(
    weighted_train_default,
    factors=DEFAULT_PARAMS["factors"],
    regularization=DEFAULT_PARAMS["regularization"],
    alpha=DEFAULT_PARAMS["alpha"],
    iterations=DEFAULT_PARAMS["iterations"],
    random_state=RANDOM_STATE,
    show_progress=False,
)

print("Model source:", getattr(baseline_model, "source_library", "implicit"))
baseline_metrics, baseline_user_metrics = evaluate_recommendations(
    baseline_model,
    train_matrix,
    test_matrix,
    k=TOP_N_RECOMMENDATIONS,
)
display(pd.DataFrame([baseline_metrics]))

## Cell 7: Hyperparameter Search with Optuna

Optuna tests different model settings and keeps the settings that perform
best on the held-out data.

This search tunes:

- number of ALS latent factors
- regularization
- alpha confidence scaling
- ALS iterations
- BM25 `k1`
- BM25 `b`

In [ ]:
try:
    best_params, trials_df = run_optuna_search(
        train_matrix,
        test_matrix,
        n_trials=N_OPTUNA_TRIALS,
        k=TOP_N_RECOMMENDATIONS,
        random_state=RANDOM_STATE,
    )
    search_used_optuna = True
    trials_df.to_csv(OUTPUT_DIR / "als_optuna_trials.csv", index=False)
    print("Optuna search finished.")
except ImportError as exc:
    print(exc)
    print("Using default parameters so the rest of the notebook can run.")
    best_params = DEFAULT_PARAMS.copy()
    trials_df = pd.DataFrame()
    search_used_optuna = False

pd.DataFrame([best_params]).to_csv(OUTPUT_DIR / "best_als_parameters.csv", index=False)
print("Selected parameters:")
display(pd.DataFrame([best_params]))

if not trials_df.empty:
    display(trials_df.sort_values("value", ascending=False).head())

## Cell 8: Train the Final ALS Model

This cell retrains ALS using the selected hyperparameters. If Optuna ran,
these are the best parameters found during the search. Otherwise, they are
the default teaching parameters.

In [ ]:
weighted_train = apply_bm25_weighting(
    train_matrix,
    k1=float(best_params["bm25_k1"]),
    b=float(best_params["bm25_b"]),
)

final_model = fit_als_model(
    weighted_train,
    factors=int(best_params["factors"]),
    regularization=float(best_params["regularization"]),
    alpha=float(best_params["alpha"]),
    iterations=int(best_params["iterations"]),
    random_state=RANDOM_STATE,
    show_progress=False,
)

final_metrics, user_level_metrics = evaluate_recommendations(
    final_model,
    train_matrix,
    test_matrix,
    k=TOP_N_RECOMMENDATIONS,
)

pd.DataFrame([final_metrics]).to_csv(OUTPUT_DIR / "als_evaluation_metrics.csv", index=False)
user_level_metrics.to_csv(OUTPUT_DIR / "als_user_level_metrics.csv", index=False)

print("Final model source:", getattr(final_model, "source_library", "implicit"))
display(pd.DataFrame([final_metrics]))

## Cell 9: Create Product Recommendations

This cell creates top product recommendations for a few sample customers.
The model filters out products the customer has already interacted with in
the training data.

In [ ]:
sample_users = list(mappings.user_to_index.keys())[:5]

recommendation_tables = []
for user_id in sample_users:
    user_recommendations = get_recommendations_for_user(
        final_model,
        user_id,
        mappings,
        train_matrix,
        catalog,
        n_recommendations=TOP_N_RECOMMENDATIONS,
    )
    user_recommendations.insert(0, "user_id", user_id)
    recommendation_tables.append(user_recommendations)

all_recommendations = pd.concat(recommendation_tables, ignore_index=True)
all_recommendations.to_csv(
    OUTPUT_DIR / "sample_user_recommendations.csv", index=False
)

display(all_recommendations.head(15))

## Cell 10: Find Similar Products

ALS learns a vector for each product. Products with similar vectors are
products that tend to be liked by similar customers.

In [ ]:
sample_item_id = catalog["item_id"].iloc[0]
similar_items = get_similar_items(
    final_model,
    sample_item_id,
    mappings,
    catalog,
    n_items=10,
)
similar_items.to_csv(OUTPUT_DIR / "sample_similar_items.csv", index=False)

print("Similar products for:", sample_item_id)
display(similar_items)

## Cell 11: Visual Summary

This cell saves two simple charts:

- category mix of the sample recommendations
- Optuna trial history, if Optuna was installed and used

In [ ]:
category_counts = all_recommendations["category"].value_counts().sort_values()

plt.figure(figsize=(8, 5))
category_counts.plot(kind="barh", color="#287c76")
plt.title("Category Mix in Sample Recommendations")
plt.xlabel("Recommendation Count")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "recommendation_category_mix.png", dpi=150)
plt.show()

if not trials_df.empty and "value" in trials_df.columns:
    plt.figure(figsize=(8, 4))
    plt.plot(trials_df["number"], trials_df["value"], marker="o", color="#455a64")
    plt.title("Optuna Trial History")
    plt.xlabel("Trial Number")
    plt.ylabel(f"Recall@{TOP_N_RECOMMENDATIONS}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "optuna_trial_history.png", dpi=150)
    plt.show()
else:
    print("Optuna trial chart skipped because Optuna did not run.")

## Cell 12: What to Try Next

After you understand the sample notebook, try these changes:

- Increase `N_OPTUNA_TRIALS` for a more thorough search.
- Change event weights in `code/product_recommender/data.py`.
- Add more interaction rows to `customer_product_interactions.csv`.
- Compare recommendations for customers from different segments.
- Add cold-start logic for new products with no interactions.

The output files are saved in the `outputs/` folder.